In [ ]:
from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch
import numpy as np
import os
import sys
import umap
import plotly.express as px
import torch

sys.path.append(os.path.abspath("/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models"))
# Register custom resolver to handle multiplication in OmegaConf interpolation
OmegaConf.register_new_resolver("mul", lambda x, y: float(x) * float(y))
OmegaConf.register_new_resolver("div", lambda a, b: float(a) / float(b))

In [ ]:
# snapshot_dir = "/storage/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments_0908/lam_agibot_egodex_droid_xatten_noad_t/2025-09-14/00-06-25/0"
# snapshot_dir = '/storage/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments_0908/lam_agibot_egodex_droid_mt_xatten_t/2025-09-09/18-31-48/0'
snapshot_dir = "/storage/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments_0908/eam_agibot_egodex_droid_mt_t/2025-09-09/18-25-21/0"
config_path = f"{snapshot_dir}/.hydra"

print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")
model = instantiate(cfg.model)
model = model.cuda().eval()
snapshot = torch.load(f"{snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
model.load_state_dict(snapshot["model"])
val_dataloader = instantiate(cfg.val_data_loader)

In [ ]:
bs_snapshot_dir = "/storage/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments_0908/eam_agibot_egodex_droid_mt_t/2025-09-09/18-25-21/0"
bs_config_path = f"{bs_snapshot_dir}/.hydra"
print(bs_config_path)

bs_cfg = OmegaConf.load(bs_config_path + "/config.yaml")
bs_model = instantiate(bs_cfg.model)
bs_model = bs_model.cuda().eval()
bs_snapshot = torch.load(f"{bs_snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
bs_model.load_state_dict(bs_snapshot["model"])

In [ ]:
# cfg.val_data_loader[0].dataset.datasets['Droid'].transform.sample_size = 64
# cfg.val_data_loader[1].dataset.datasets['EgoDex'].transform.sample_size = 64
# cfg.val_data_loader[3].dataset.datasets['MPK'].transform.sample_size = 64
# val_dataloader = instantiate(cfg.val_data_loader)
agibot_dataloader = val_dataloader[0]
egodex_dataloader = val_dataloader[1]
droid_dataloader = val_dataloader[2]

In [ ]:
droid_zs = []
for i, droid_batch in enumerate(droid_dataloader):
    rgb = droid_batch["rgb"].cuda()
    actions = droid_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions,
                                   morphology_index= droid_batch["morphology_index"].cuda(),
                                   ee_action_dim= droid_batch["ee_action_dim"].cuda(),
                                     ).detach().cpu().numpy()
    droid_zs.append(z)
    if i > 100:
        break

egodex_zs = []
for i, egodex_batch in enumerate(egodex_dataloader):
    rgb = egodex_batch["rgb"].cuda()
    actions = egodex_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions,
                                    morphology_index= egodex_batch["morphology_index"].cuda(),
                                    ee_action_dim=egodex_batch["ee_action_dim"].cuda() 
                                     ).detach().cpu().numpy()
    egodex_zs.append(z)
    if i > 100:
        break

agibot_zs = []
for i, agibot_batch in enumerate(agibot_dataloader):
    rgb = agibot_batch["rgb"].cuda()
    actions = agibot_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions,
                                     morphology_index=agibot_batch["morphology_index"].cuda(),
                                    ee_action_dim=agibot_batch["ee_action_dim"].cuda() 
                                    ).detach().cpu().numpy()
    agibot_zs.append(z)
    if i > 100:
        break

In [ ]:
droid_zs_c = np.concatenate(droid_zs, axis=0)
egodex_zs_c = np.concatenate(egodex_zs, axis=0)
agibot_zs_c = np.concatenate(agibot_zs, axis=0)

In [ ]:
flatten_droid_zs = rearrange(droid_zs_c, "b t a m -> (b t) (a m)")
flatten_egodex_zs = rearrange(egodex_zs_c, "b t a m -> (b t) (a m)")
flatten_agibot_zs = rearrange(agibot_zs_c, "b t a m -> (b t) (a m)")

In [ ]:
embeddings = np.stack([flatten_droid_zs, flatten_egodex_zs, flatten_agibot_zs], axis=0)
# embeddings = np.stack([flatten_droid_zs, flatten_egodex_zs], axis=0)


In [ ]:
def plot_umap_3d_interactive(embeddings, color_labels=None):
    """
    embeddings: torch.Tensor or np.ndarray of shape (S, B, M)
    color_labels: optional array of shape (S*B,) for coloring
    """
    # Convert to numpy
    if torch.is_tensor(embeddings):
        embeddings = embeddings.detach().cpu().numpy()

    S, B, M = embeddings.shape
    embeddings_flat = embeddings.reshape(S * B, M)

    # Default color labels = data source index
    if color_labels is None:
        color_labels = np.repeat(np.arange(S), B)

    # Run UMAP
    reducer = umap.UMAP(n_components=3, random_state=42)
    embeddings_3d = reducer.fit_transform(embeddings_flat)

    # Create interactive plot
    fig = px.scatter_3d(
        x=embeddings_3d[:, 0],
        y=embeddings_3d[:, 1],
        z=embeddings_3d[:, 2],
        color=color_labels.astype(str),  # plotly requires string or category
        labels={'color': 'Data Source'},
        title="Interactive 3D UMAP"
    )
    fig.update_traces(marker=dict(size=3, opacity=0.7))
    # fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
    fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=30),
    scene=dict(
        xaxis=dict(showticklabels=False),
        yaxis=dict(showticklabels=False),
        zaxis=dict(showticklabels=False)
    )
    )
    fig.write_image("umap_3d_canonical_view.png", width=1920, height=1080, scale=2)
    fig.show()


In [ ]:
S, B, M = embeddings.shape
print(S, B, M)

In [ ]:
plot_umap_3d_interactive(embeddings, color_labels=np.repeat(np.arange(embeddings.shape[0]), embeddings.shape[1]))

## Transfer

In [ ]:
droid_iter = iter(droid_dataloader)  # Create an iterator
egodex_iter = iter(egodex_dataloader)  # Create an iterator
agibot_iter = iter(agibot_dataloader)  # Create an iterator
batch_idx=0

In [ ]:
droid_batch = next(droid_iter)
egodex_batch = next(egodex_iter)
agibot_batch = next(agibot_iter)
batch_idx+=1

In [ ]:
droid_rgb = droid_batch["rgb"].cuda()
egodex_rgb = egodex_batch["rgb"].cuda()
agibot_rgb = agibot_batch["rgb"].cuda()

In [ ]:
print(batch_idx)

In [ ]:
with torch.no_grad():
    droid_x = model._forward_tokenizer_encode(droid_rgb)
    droid_xhat = droid_x.detach().clone()
    egodex_x = model._forward_tokenizer_encode(egodex_rgb).detach().clone()
    egodex_xhat = egodex_x.detach().clone()
    agibot_x = model._forward_tokenizer_encode(agibot_rgb).detach().clone()
    agibot_xhat = agibot_x.detach().clone()
    
    # bs_droid_x = bs_model._forward_tokenizer_encode(droid_rgb)
    # bs_droid_xhat = bs_droid_x.detach().clone()
    # bs_egodex_x = bs_model._forward_tokenizer_encode(egodex_rgb).detach().clone()
    # bs_egodex_xhat = bs_egodex_x.detach().clone()
    # bs_agibot_x = bs_model._forward_tokenizer_encode(agibot_rgb).detach().clone()
    # bs_agibot_xhat = bs_agibot_x.detach().clone()

In [ ]:

with torch.no_grad():
    # droid_action_latent = model._forward_inverse_model(droid_x, droid_batch["actions"].cuda())
    # egodex_action_latent = model._forward_inverse_model(egodex_x, egodex_batch["actions"].cuda())
    # agibot_action_latent = model._forward_inverse_model(agibot_x, agibot_batch["actions"].cuda())
    
    droid_action_latent = model._forward_inverse_model(droid_x, droid_batch["actions"].cuda(),  morphology_index = droid_batch["morphology_index"].cuda(),
                                   ee_action_dim = droid_batch["ee_action_dim"].cuda())
    egodex_action_latent = model._forward_inverse_model(egodex_x, egodex_batch["actions"].cuda(),  morphology_index = droid_batch["morphology_index"].cuda(),
                                   ee_action_dim = droid_batch["ee_action_dim"].cuda())
    agibot_action_latent = model._forward_inverse_model(agibot_x, agibot_batch["actions"].cuda(),  morphology_index = droid_batch["morphology_index"].cuda(),
                                   ee_action_dim = droid_batch["ee_action_dim"].cuda())
    
    # bs_droid_action_latent = bs_model._forward_inverse_model(bs_droid_x, droid_batch["actions"].cuda(),  morphology_index = droid_batch["morphology_index"].cuda(),
    #                                ee_action_dim = droid_batch["ee_action_dim"].cuda())
    # bs_egodex_action_latent = bs_model._forward_inverse_model(bs_egodex_x, egodex_batch["actions"].cuda(),  morphology_index = droid_batch["morphology_index"].cuda(),
    #                                ee_action_dim = droid_batch["ee_action_dim"].cuda())
    # bs_agibot_action_latent = bs_model._forward_inverse_model(bs_agibot_x, agibot_batch["actions"].cuda(),  morphology_index = droid_batch["morphology_index"].cuda(),
    #                                ee_action_dim = droid_batch["ee_action_dim"].cuda())

    # egodex_rgbhat = model._encode_decode(egodex_rgb).detach().cpu().numpy()
    # agibot_rgbhat = model._encode_decode(agibot_rgb).detach().cpu().numpy()
    
    # bs_egodex_rgbhat = bs_model._encode_decode(egodex_rgb).detach().cpu().numpy()
    # bs_agibot_rgbhat = bs_model._encode_decode(agibot_rgb).detach().cpu().numpy()

    # #egodex to droid
    # T = droid_xhat.shape[1]
    # for idx in range(1, T):
    #     droid_xhat[:, idx] = model._forward_forward_model(
    #         droid_xhat[:, :idx], egodex_action_latent[:, 1 : idx + 1], morphology_index=droid_batch["morphology_index"].cuda()
    #     )[:, -1]
    # egodex2droid_rgbhat = model._forward_tokenizer_decode(droid_xhat, droid_rgb.shape).detach().cpu().numpy()
    
    # #droid to egodex
    # T = droid_xhat.shape[1]
    # for idx in range(1, T):
    #     droid_xhat[:, idx] = model._forward_forward_model(
    #         droid_xhat[:, :idx], egodex_action_latent[:, 1 : idx + 1], morphology_index=droid_batch["morphology_index"].cuda()
    #     )[:, -1]
    # egodex2droid_rgbhat = model._forward_tokenizer_decode(droid_xhat, droid_rgb.shape).detach().cpu().numpy()


    # # egodex to agibot
    # T = agibot_xhat.shape[1]
    # for idx in range(1, T):
    #     agibot_xhat[:, idx] = model._forward_forward_model(
    #         agibot_xhat[:, :idx], egodex_action_latent[:, 1 : idx + 1], morphology_index=agibot_batch["morphology_index"].cuda()
    #     )[:, -1]
    # egodex2agibot_rgbhat = model._forward_tokenizer_decode(agibot_xhat, agibot_rgb.shape).detach().cpu().numpy()

    # # agibot to egodex
    # T = egodex_xhat.shape[1]
    # for idx in range(1, T):
    #     egodex_xhat[:, idx] = model._forward_forward_model(
    #         egodex_xhat[:, :idx], agibot_action_latent[:, 1 : idx + 1], morphology_index=egodex_batch["morphology_index"].cuda()
    #     )[:, -1]
    # agibot2egodex_rgbhat = model._forward_tokenizer_decode(egodex_xhat, egodex_rgb.shape).detach().cpu().numpy()
    
    
    # egodex to agibot
    # T = bs_agibot_xhat.shape[1]
    # for idx in range(1, T):
    #     bs_agibot_xhat[:, idx] = bs_model._forward_forward_model(
    #         bs_agibot_xhat[:, :idx], bs_egodex_action_latent[:, 1 : idx + 1], morphology_index=agibot_batch["morphology_index"].cuda()
    #     )[:, -1]
    # bs_egodex2agibot_rgbhat = bs_model._forward_tokenizer_decode(bs_agibot_xhat, agibot_rgb.shape).detach().cpu().numpy()

    # # agibot to egodex
    # T = bs_egodex_xhat.shape[1]
    # for idx in range(1, T):
    #     bs_egodex_xhat[:, idx] = bs_model._forward_forward_model(
    #         bs_egodex_xhat[:, :idx], bs_agibot_action_latent[:, 1 : idx + 1], morphology_index=egodex_batch["morphology_index"].cuda()
    #     )[:, -1]
    # bs_agibot2egodex_rgbhat = bs_model._forward_tokenizer_decode(bs_egodex_xhat, egodex_rgb.shape).detach().cpu().numpy()

In [ ]:
with torch.no_grad():
    #egodex to droid
    droid_xhat = droid_x.detach().clone()
    T = droid_xhat.shape[1]
    for idx in range(1, T):
        droid_xhat[:, idx] = model._forward_forward_model(
            droid_xhat[:, :idx], egodex_action_latent[:, 1 : idx + 1], morphology_index=droid_batch["morphology_index"].cuda()
        )[:, -1]
    egodex2droid_rgbhat = model._forward_tokenizer_decode(droid_xhat, droid_rgb.shape).detach().cpu().numpy()
    
    #droid to egodex
    egodex_xhat = egodex_x.detach().clone()
    T = egodex_xhat.shape[1]
    for idx in range(1, T):
        egodex_xhat[:, idx] = model._forward_forward_model(
            egodex_xhat[:, :idx], droid_action_latent[:, 1 : idx + 1], morphology_index=egodex_batch["morphology_index"].cuda()
        )[:, -1]
    droid2egodex_rgbhat = model._forward_tokenizer_decode(egodex_xhat, egodex_rgb.shape).detach().cpu().numpy()


    # egodex to agibot
    agibot_xhat = agibot_x.detach().clone()
    T = agibot_xhat.shape[1]
    for idx in range(1, T):
        agibot_xhat[:, idx] = model._forward_forward_model(
            agibot_xhat[:, :idx], egodex_action_latent[:, 1 : idx + 1], morphology_index=agibot_batch["morphology_index"].cuda()
        )[:, -1]
    egodex2agibot_rgbhat = model._forward_tokenizer_decode(agibot_xhat, agibot_rgb.shape).detach().cpu().numpy()

    # agibot to egodex
    egodex_xhat = egodex_x.detach().clone()
    T = egodex_xhat.shape[1]
    for idx in range(1, T):
        egodex_xhat[:, idx] = model._forward_forward_model(
            egodex_xhat[:, :idx], agibot_action_latent[:, 1 : idx + 1], morphology_index=egodex_batch["morphology_index"].cuda()
        )[:, -1]
    agibot2egodex_rgbhat = model._forward_tokenizer_decode(egodex_xhat, egodex_rgb.shape).detach().cpu().numpy()
    
    # agibot to droid
    droid_xhat = droid_x.detach().clone()
    T = droid_xhat.shape[1]
    for idx in range(1, T):
        droid_xhat[:, idx] = model._forward_forward_model(
            droid_xhat[:, :idx], agibot_action_latent[:, 1 : idx + 1], morphology_index=droid_batch["morphology_index"].cuda()
        )[:, -1]
    agibot2droid_rgbhat = model._forward_tokenizer_decode(droid_xhat, egodex_rgb.shape).detach().cpu().numpy()
    
    #droid to agibot
    agibot_xhat = agibot_x.detach().clone()
    T = agibot_xhat.shape[1]
    for idx in range(1, T):
        agibot_xhat[:, idx] = model._forward_forward_model(
            agibot_xhat[:, :idx], droid_action_latent[:, 1 : idx + 1], morphology_index=agibot_batch["morphology_index"].cuda()
        )[:, -1]
    droid2agibot_rgbhat = model._forward_tokenizer_decode(agibot_xhat, egodex_rgb.shape).detach().cpu().numpy()

In [ ]:
egodex_rgb = egodex_rgb.detach().cpu().numpy()
agibot_rgb = agibot_rgb.detach().cpu().numpy()
droid_rgb = droid_rgb.detach().cpu().numpy()

In [ ]:
import mediapy as media
save_path = f"/storage/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/scripts/transfer_res/transfer_results_eam_batch{batch_idx}"
os.makedirs(save_path, exist_ok=True)
for i in range(egodex_rgb.shape[0]):
    cur_save_path = save_path + f'/{i}'
    os.makedirs(cur_save_path, exist_ok=True)
    
    media.write_video(f"{cur_save_path}/egodex_refer.mp4", rearrange(egodex_rgb[i], "t c h w -> t h w c") , fps=10)
    media.write_video(f"{cur_save_path}/egodex2agibot.mp4", rearrange(egodex2agibot_rgbhat[i], "t c h w -> t h w c") , fps=10)
    media.write_video(f"{cur_save_path}/egodex2droid.mp4", rearrange(egodex2droid_rgbhat[i], "t c h w -> t h w c") , fps=10)
    
    # media.write_video(f"{cur_save_path}/bs_egodex2agibot.mp4", rearrange(bs_egodex2agibot_rgbhat[i], "t c h w -> t h w c") , fps=10)
    
    media.write_video(f"{cur_save_path}/agibot_refer.mp4", rearrange(agibot_rgb[i], "t c h w -> t h w c") , fps=10)
    media.write_video(f"{cur_save_path}/agibot2egodex.mp4", rearrange(agibot2egodex_rgbhat[i], "t c h w -> t h w c") , fps=10)
    media.write_video(f"{cur_save_path}/agibot2droid.mp4", rearrange(agibot2droid_rgbhat[i], "t c h w -> t h w c") , fps=10)
    # media.write_video(f"{cur_save_path}/bs_agibot2egodex.mp4", rearrange(bs_agibot2egodex_rgbhat[i], "t c h w -> t h w c") , fps=10)
    
    media.write_video(f"{cur_save_path}/droid_refer.mp4", rearrange(droid_rgb[i], "t c h w -> t h w c") , fps=10)
    media.write_video(f"{cur_save_path}/droid2egodex.mp4", rearrange(droid2egodex_rgbhat[i], "t c h w -> t h w c") , fps=10)
    media.write_video(f"{cur_save_path}/droid2agibot.mp4", rearrange(droid2agibot_rgbhat[i], "t c h w -> t h w c") , fps=10)

In [ ]:
print(idx)